In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
#import matplotlib.pyplot as plt
import random

#connect to google sheet
from google.colab import auth
import gspread
from google.auth import default

#store embeddings
import json


In [ ]:
#authenticate and connect to Google Sheets
auth.authenticate_user()

creds, _ = default()
gc = gspread.authorize(creds)

#model
model = SentenceTransformer("all-MiniLM-L6-v2")

#read spreadsheet from Google Sheets
flowers = "Victorian Flowers"

sh = gc.open(flowers)

worksheet = sh.sheet1

data = worksheet.get_all_records()  #reads headers automatically
df = pd.DataFrame(data)

#check that the columns are correct
assert "Flower" in df.columns and "Meaning" in df.columns, "Sheet must have 'Flower' and 'Meaning' columns"

#generate embeddings for flowers (convert meaning to vector)
meanings = df["Meaning"].tolist()

flower_embeddings = model.encode(
    meanings,
    normalize_embeddings=True
)

#store each embedding as a JSON string to prep for CSV
df["meaning_embedding"] = [json.dumps(embedding.tolist()) for embedding in flower_embeddings]

#export to CSV
output_path = "victorian_flowers_with_embeddings.csv"
df.to_csv(output_path, index=False)

print(f"Saved {len(df)} rows to {output_path}")


"""
#test flower list
flowers = [
    {
        "flower": "Chrysanthemum",
        "meaning": "I love you"
    },
    {
        "flower": "Purple Hyacinth",
        "meaning": "sad"
    },
    {
        "flower": "Snapdragona",
        "meaning": "lies"
    },
    {
        "flower": "Goldenrod",
        "meaning": "luck"
    }
]

#print(flowers)

#generate embeddings for flowers (convert meaning to vector)
meanings = [flower["meaning"] for flower in flowers]

flower_embeddings = model.encode(
    meanings,
    normalize_embeddings=True
)
"""



In [ ]:
#load model
model = SentenceTransformer("all-MiniLM-L6-v2")

#load flower embeddings
df = pd.read_csv("victorian_flowers_with_embeddings.csv")

#convert the JSON string back into a real list of numbers
df["meaning_embedding"] = df["meaning_embedding"].apply(json.loads)

#stack all embeddings into one matrix (for cosine_similarity)
flower_embeddings_matrix = np.stack(df["meaning_embedding"].values)

#ask for input
phrase = input('Please enter your phrase: ')
#print(phrase)

#generate embeddings for input
phrase_embedding = model.encode(
    phrase,
    normalize_embeddings=True
)

#use cosine similarity to measure the angle between two vectors
scores = cosine_similarity(
    [phrase_embedding],
    flower_embeddings_matrix
)[0]

#pair each flower with its score and sort high -> low
results = []

for i, score in enumerate(scores):
    results.append((df.iloc[i]["Flower"], df.iloc[i]["Meaning"], score))

results.sort(key=lambda x: x[2], reverse=True)

#top 5 flowers and scores
top_results = results[:5]

top_scores = [score for _, _, score in top_results]

#calculate each flower's proportional share of 10 and converts to whole number
total_score = sum(top_scores)
raw_shares = [(score / total_score) * 10 for score in top_scores]

counts = [int(share) for share in raw_shares]

#figure out how many flowers are still needed to reach 10
remainder = 10 - sum(counts)

#sort by leftover fraction (largest first) and hand out remaining flowers
fractional_parts = [(i, raw_shares[i] - counts[i]) for i in range(len(raw_shares))]
fractional_parts.sort(key=lambda x: x[1], reverse=True)

for i in range(remainder):
    index = fractional_parts[i][0]
    counts[index] += 1

flower_emoji_map = {
    "Abatina": "🌸",
    "Acanthus": "🌿",
    "Aloe": "🌿",
    "Amaryllis": "🌺",
    "Anemone": "🌸",
    "Angelica": "🌿",
    "Apple blossom": "🍎",
    "Arborvitae": "🌿",
    "Aster": "🌼",
    "Baby’s breath": "🍀",
    "Bachelor’s button": "🪻",
    "Basil": "🌿",
    "Bay tree": "🌿",
    "Begonia": "🌸",
    "Belledonna": "🪻",
    "Bittersweet": "🌿",
    "Black-eyed Susan": "🌼",
    "Bluebell": "🪻",
    "Borage": "🪻",
    "Butterfly weed": "🌺",
    "Calla lily": "🌷",
    "Camellia, pink": "🌸",
    "Camellia, red": "🌹",
    "Camellia, white": "🌸",
    "Candytuft": "🍭",
    "Carnation": "🌸",
    "Red carnation": "🌹",
    "White carnation": "🌸",
    "Pink carnation": "🌸",
    "Striped": "🌸",
    "Yellow carnation": "🌼",
    "Chamomile": "🌼",
    "Chives": "🌿",
    "Chrysanthemum, red": "🌹",
    "Chrysanthemum, yellow": "🌼",
    "Chrysanthemum, white": "🌼",
    "Clematis": "🪻",
    "Clematis, evergreen": "🌿",
    "Clover, white": "🌿",
    "Columbine": "🪻",
    "Columbine, purple": "🪻",
    "Columbine, red": "🌹",
    "Coreopsis": "🌼",
    "Coriander": "🌿",
    "Crab blossom": "🦀",
    "Crocus, spring": "🪻",
    "Cyclamen": "🌸",
    "Daffodil": "🌼",
    "Dahlia, single": "🌸",
    "Daisy": "🌼",
    "Daylily": "🌷",
    "Dill": "🌿",
    "Edelweiss": "🌸",
    "Fennel": "🌿",
    "Fern": "🌿",
    "Forget-me-not": "🪻",
    "Gardenia": "🌸",
    "Geranium": "🌸",
    "Gladiolus": "🌺",
    "Goldenrod": "🌻",
    "Heliotrope": "🪻",
    "Hibiscus": "🌺",
    "Holly": "🌿",
    "Hollyhock": "🌺",
    "Honeysuckle": "🍯",
    "Hyacinth": "🪻",
    "Blue Hyacinth": "🪻",
    "Purple Hyacinth": "🪻",
    "Yellow Hyacinth": "🌼",
    "White Hyacinth": "🌸",
    "Hydrangea": "🪻",
    "Hyssop": "🌿",
    "Iris": "🪻",
    "Ivy": "🌿",
    "Jasmine, white": "🌸",
    "Jasmine, yellow": "🌼",
    "Lady’s Slipper": "🌷",
    "Larkspur": "🪻",
    "Lavender": "🪻",
    "Lemon balm": "🍋",
    "Lilac": "🪻",
    "Lily (white)": "🌷",
    "Lily (yellow)": "🌷",
    "Lily (orange)": "🌷",
    "Lily, tiger": "🌷",
    "Lily-of-the-valley": "🌸",
    "Lotus Flower": "🪷",
    "Magnolia": "🌸",
    "Marigold": "🌼",
    "Marjoram": "🌿",
    "Mint": "🌿",
    "Morning glory": "🪻",
    "Myrtle": "🌿",
    "Nasturtium": "🌺",
    "Oak": "🌿",
    "Oregano": "🌿",
    "Pansy": "🪻",
    "Parsley": "🌿",
    "Peony": "🌸",
    "Pine": "🌿",
    "Poppy": "🌹",
    "Rhododendron": "🌺",
    "Rose, red": "🌹",
    "Rose, dark crimson": "🌹",
    "Rose, pink": "🌹",
    "Rose, white": "🌹",
    "Rose, yellow": "🌹",
    "Rosemary": "🌿",
    "Rue": "🌿",
    "Sage": "🌿",
    "Salvia, blue": "🪻",
    "Salvia, red": "🌹",
    "Savory": "🌿",
    "Snapdragon": "🌷",
    "Sorrel": "🌿",
    "Southernwood": "🌿",
    "Spearmint": "🌿",
    "Speedwell": "🪻",
    "Sunflower, dwarf": "🌻",
    "Sunflower, tall": "🌻",
    "Sweet pea": "🌸",
    "Sweet William": "🌸",
    "Sweet woodruff": "🌿",
    "Tansy": "🌼",
    "Tarragon": "🌿",
    "Thyme": "🌿",
    "Tulip, red": "🌷",
    "Tulip, yellow": "🌷",
    "Valerian": "🌸",
    "Violet": "🪻",
    "Willow": "🌿",
    "Yarrow": "🌼",
    "Zinnia": "🌸",
}

#build bouquet
print("")
print("~~~*~~~*~~~*~~~*~~~*~~~")
print("")
print(f"Your '{phrase}' Bouquet:\n")

# Print flower list
bouquet_emojis = []

for (flower, meaning, score), count in zip(top_results, counts):
    if count > 0:
        emoji = flower_emoji_map.get(flower, "🌸")
        print(f"{count} × {flower}")

        # Add the correct number of emojis to the bouquet
        bouquet_emojis.extend([emoji] * count)

# Randomise bouquet arrangement
random.shuffle(bouquet_emojis)

print("\nYour Bouquet:\n")

# Bouquet shape: 3-4-3
shape = [3, 4, 3]
index = 0

for n in shape:
    row = bouquet_emojis[index:index+n]

    # centre each row around the middle row
    print(" " * ((4 - n) * 2) + " ".join(row))
    index += n

print("\n~~~*~~~*~~~*~~~*~~~*~~~")